# AMD — análise financeira

Notebook para Google Colab: obtém dados históricos da AMD, apresenta demonstrações financeiras e cria gráficos com uma apresentação mais cuidada.

**Não exporta CSV nem ficheiros.** Os resultados são apresentados diretamente no notebook.

> Dados via Yahoo Finance (`yfinance`), sujeitos a disponibilidade e atraso. Conteúdo informativo, não aconselhamento financeiro.

In [ ]:
%pip -q install yfinance pandas matplotlib

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

TICKER = "AMD"
PERIODO = "5y"       # exemplos: "1y", "2y", "5y", "max"
INTERVALO = "1d"     # exemplos: "1d", "1wk", "1mo"

plt.style.use("seaborn-v0_8-whitegrid")
amd = yf.Ticker(TICKER)
precos = amd.history(period=PERIODO, interval=INTERVALO, auto_adjust=False)

if precos.empty:
    raise RuntimeError("Sem preços disponíveis. Confirma a ligação e os parâmetros.")

print(f"Empresa: {TICKER} · Período: {PERIODO} · Registos: {len(precos):,}")
display(precos.tail())

## 1. Evolução do preço da ação

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), dpi=120)
fecho = precos["Close"].dropna()
ax.plot(fecho.index, fecho, color="#2563eb", linewidth=2.4, label="Preço de fecho")
ax.fill_between(fecho.index, fecho.astype(float), color="#60a5fa", alpha=0.16)
if len(fecho) > 0:
    ax.scatter(fecho.index[-1], fecho.iloc[-1], color="#1d4ed8", s=45, zorder=3)
    ax.annotate(f"${fecho.iloc[-1]:.2f}", (fecho.index[-1], fecho.iloc[-1]),
                xytext=(-10, 12), textcoords="offset points", ha="right",
                fontsize=10, fontweight="bold")
ax.set_title(f"{TICKER} · Evolução do preço de fecho", fontsize=17, fontweight="bold", pad=16)
ax.set_xlabel("Data")
ax.set_ylabel("Preço (USD)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, alpha=0.22)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 2. Volume negociado

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5), dpi=120)
ax.fill_between(precos.index, precos["Volume"], color="#0f766e", alpha=0.72, linewidth=0)
ax.set_title(f"{TICKER} · Volume negociado", fontsize=16, fontweight="bold", pad=14)
ax.set_xlabel("Data")
ax.set_ylabel("Número de ações")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis="y", alpha=0.22)
fig.tight_layout()
plt.show()

## 3. Demonstrações financeiras

In [ ]:
demonstracoes = {
    "Resultados anuais": amd.income_stmt,
    "Resultados trimestrais": amd.quarterly_income_stmt,
    "Balanço anual": amd.balance_sheet,
    "Balanço trimestral": amd.quarterly_balance_sheet,
    "Fluxo de caixa anual": amd.cashflow,
    "Fluxo de caixa trimestral": amd.quarterly_cashflow,
}
for nome, tabela in demonstracoes.items():
    print(f"\n### {nome}")
    if tabela is not None and not tabela.empty:
        display(tabela)
    else:
        print("Sem dados disponíveis.")

## 4. Receita e lucro líquido

In [ ]:
resultados = amd.income_stmt
if resultados is not None and not resultados.empty:
    linhas = {str(indice).lower(): indice for indice in resultados.index}
    receita_key = next((v for k, v in linhas.items() if "total revenue" in k or k == "revenue"), None)
    lucro_key = next((v for k, v in linhas.items() if "net income" in k), None)
    series = {}
    if receita_key is not None:
        series["Receita"] = pd.to_numeric(resultados.loc[receita_key], errors="coerce")
    if lucro_key is not None:
        series["Lucro líquido"] = pd.to_numeric(resultados.loc[lucro_key], errors="coerce")
    if series:
        financeiro = pd.DataFrame(series).sort_index()
        display(financeiro)
        fig, ax = plt.subplots(figsize=(12, 5.5), dpi=120)
        financeiro.plot(kind="bar", ax=ax, color=["#2563eb", "#f59e0b"][:len(financeiro.columns)], width=0.72)
        ax.set_title(f"{TICKER} · Receita e lucro líquido anual", fontsize=16, fontweight="bold", pad=14)
        ax.set_xlabel("Período")
        ax.set_ylabel("USD")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.22)
        ax.legend(frameon=False)
        plt.xticks(rotation=0)
        fig.tight_layout()
        plt.show()
    else:
        print("Não foi possível identificar receita/lucro nos dados disponíveis.")
else:
    print("Demonstração de resultados não disponível.")

## 5. Indicadores gerais

In [ ]:
try:
    info = amd.get_info()
    chaves = ["longName", "sector", "industry", "currency", "marketCap",
              "trailingPE", "forwardPE", "priceToBook", "profitMargins",
              "returnOnEquity", "dividendYield"]
    resumo = {chave: info.get(chave) for chave in chaves if info.get(chave) is not None}
    display(pd.DataFrame(resumo.items(), columns=["Indicador", "Valor"]))
except Exception as erro:
    print(f"Não foi possível obter os indicadores: {erro}")